## Simulating a lock complex with two lock chambers on different edges
In this notebook, we simulate a lock on a simple one-route graph which randomly generated vessels have to pass. We add a lock complex object on the graph, which has two lock chambers: a small one and a large one. The lock chambers are located next to each other, and we simplify this by saying that the lock chambers are on top of each other. Vessels are locked together if they can fit inside the lock and register themselves before the start of the operation.

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform
from shapely import reverse

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.utils import create_object
from opentnsim.utils import generate_vessels_from_distribution
from opentnsim.graph.visualizations import plot_graph
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


In [2]:
%load_ext autoreload
%autoreload 2

#### 0. Create environment

In [3]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph
We create a graph with two edges between a pair of nodes. To faciliate this, we create a MultiDiGraph.

In [4]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.MultiDiGraph()

# add nodes
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-25000,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(25000,0)))

# add edges
graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-25000, 0),Point(-5000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-25000, 0)])), weight=1, length_m=25000-5000)

northern_branch = LineString([Point(-5000, 0),Point(-4500, 50),Point(4500, 50),Point(5000, 0)])
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, northern_branch), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, reverse(northern_branch)), weight=1, length_m=10000)

southern_branch = LineString([Point(-5000, 0),Point(-4500, -50),Point(4500, -50),Point(5000, 0)])
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, southern_branch), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, reverse(southern_branch)), weight=1, length_m=10000)

graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(25000, 0),Point(5000, 0)])), weight=1, length_m=25000-5000)

# add graph to environment
env.graph = graph

In [5]:
plot_graph(graph)

#### 1+ Adding infrastructure
We add two lock chambers with different dimensions:

In [6]:
lock_chamber_I = IsLockChamber(env=env,
                               lock_length = 300,
                               lock_width = 40,
                               lock_depth = 6,
                               name='Lock chamber I',
                               gate_open = '0',
                               edge = ('0','1', 0),
                               geometry_m = Polygon([Point(-150, 30),Point(-150, 70),Point(150, 70),Point(150, 30)]))

lock_chamber_II = IsLockChamber(env=env,
                                lock_length = 150,
                                lock_width = 20,
                                lock_depth = 6,
                                name='Lock chamber II',
                                gate_open = '0',
                                edge = ('0','1', 1),
                                geometry_m = Polygon([Point(-75, -60),Point(-75, -40),Point(75, -40),Point(75, -60)]))

But we keep the two waiting areas, which are then used by all the vessels that have to pass the lock complex.

In [7]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1',0),
                                   distance_from_edge_start = 1000)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0',0),
                                   distance_from_edge_start = 1000)

waiting_area_C = IsLockWaitingArea(env=env,
                                   name = 'Waiting area C',
                                   edge = ('0','1',1),
                                   distance_from_edge_start = 1000)

waiting_area_D = IsLockWaitingArea(env=env,
                                   name = 'Waiting area D',
                                   edge = ('1','0',1),
                                   distance_from_edge_start = 1000)

We add the infrastructure to the complex as follows (we need to make sure that the registration nodes are before or at the nodes where the two edges split):

In [8]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber_I, lock_chamber_II],
                             waiting_areas = [waiting_area_A, waiting_area_B, waiting_area_C, waiting_area_D],
                             registration_nodes = ['0','1'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents
We again generate upstream and downstream vessel agents according to exponential distributions.

In [9]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
)

In [10]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [11]:
upstream_vessels = generate_vessels_from_distribution(env=env,
                                                      VesselClass = Vessel,
                                                      vessel_parameters = {'v':4, 'L':135, 'B':17, 'T':5, 'type':'tanker'},
                                                      mean_arrival_rate=30.,
                                                      number_of_vessels=8,
                                                      start_node = '-1',
                                                      end_node = '+1',
                                                      seed = 123)

downstream_vessels = generate_vessels_from_distribution(env=env,
                                                        VesselClass = Vessel,
                                                        vessel_parameters = {'v':4, 'L':135, 'B':17, 'T':5, 'type':'tanker'},
                                                        mean_arrival_rate=30.,
                                                        number_of_vessels=8,
                                                        start_node = '+1',
                                                        end_node = '-1',
                                                        seed = 456)

vessels = upstream_vessels + downstream_vessels

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation

In [12]:
env.run()

hi ('0', '1', 0)
hi ('0', '1', 0)


[('+1', '1', 0), ('0', '1', 0), ('0', '-1', 0)]

hi ('0', '1', 1)
hi ('0', '1', 1)


[('-1', '0', 0), ('0', '1', 1), ('1', '+1', 0)]

hi ('0', '1', 0)
hi ('0', '1', 0)


...

[('+1', '1', 0), ('0', '1', 0), ('0', '-1', 0)]

hi ('0', '1', 1)
hi ('0', '1', 1)


[('-1', '0', 0), ('0', '1', 1), ('1', '+1', 0)]

hi ('0', '1', 0)
hi ('0', '1', 0)


[('-1', '0', 0), ('0', '1', 0), ('1', '+1', 0)]

In [13]:
vessel.env.graph.edges[('1', '0', 0)]['Lock chamber']

#### 4. Inspect output

##### Gantt chart of event table
We can create a gantt chart of all the objects:

In [14]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock_chamber_I, lock_chamber_II])
fig = generate_vessel_gantt_chart(df_eventtable)

##### Time-distance diagram of vessels passing the lock and planning info
And we can create time-distance diagrams for both locks. We observe that there are lock operations with two vessels for the larger lock, while there are lock operations with only one vessel for the smaller lock. The scheduling of the vessels followed a first-come, first-served approach, with the selected lock chamber being the one that would cause the least expected delay to any individual vessel.

In [16]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber_I.plot(xlimmin = -6050, 
                          xlimmax = 6050,
                          ylimmin = pd.Timestamp('2025-01-01 01:00:00'),
                          ylimmax = pd.Timestamp('2025-01-01 09:00:00'),
                          method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

In [17]:
# We can plot the time-distance diagram
fig = lock_chamber_II.plot(xlimmin = -6050, 
                           xlimmax = 6050,
                           ylimmin = pd.Timestamp('2025-01-01 01:00:00'),
                           ylimmax = pd.Timestamp('2025-01-01 09:00:00'),
                           method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

In [ ]:
lock_complex.operation_planning